# Genie Cache & Queue — Demo

A API do Genie tem um hard limit de **5 chamadas por minuto por workspace**.
Este notebook dispara 7 queries **em paralelo** para demonstrar o problema e como o app resolve.

**O codigo e identico** — so muda a URL base.

```bash
# Pre-requisito: OAuth token
databricks auth login --host https://fevm-genie-cache-test.cloud.databricks.com
export DATABRICKS_TOKEN=$(databricks auth token | python3 -c "import sys,json; print(json.load(sys.stdin)['access_token'])")
jupyter notebook demo_notebook.ipynb
```

In [ ]:
import requests, time, json, os
from concurrent.futures import ThreadPoolExecutor, as_completed

TOKEN = os.getenv("DATABRICKS_TOKEN", "")
assert TOKEN.startswith("eyJ"), "Precisa de OAuth JWT. Veja instrucoes acima."

GENIE_HOST = "https://fevm-genie-cache-test.cloud.databricks.com"
APP_HOST   = "https://genie-cache-queue-7474650836156271.aws.databricksapps.com"
SPACE      = "01f11f1ae00114379e7671c8a4b8459f"
H = {"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"}

assert requests.get(f"{APP_HOST}/api/v1/health", headers=H).status_code == 200
requests.put(f"{APP_HOST}/api/v1/config", headers=H,
    json={"genie_space_id": SPACE, "sql_warehouse_id": "20cbd6750a2ef9ce",
          "similarity_threshold": 0.92, "cache_ttl_seconds": 86400})

questions = [
    "How many customers are there?",
    "What is the total revenue by region?",
    "Top 5 suppliers by order count",
    "Average order value per year",
    "How many parts are in the catalog?",
    "Total revenue for EUROPE region",
    "Number of orders in 1995",
]
print(f"App OK | {len(questions)} queries")

In [ ]:
# ===========================================================
# CODIGO IDENTICO para Genie direto e via App.
# So muda o base_url.
# ===========================================================

def _extract(data):
    sql, text = None, None
    for att in data.get("attachments", []):
        if isinstance(att, dict):
            if "query" in att and att["query"]: sql = att["query"].get("query")
            if "text" in att and att["text"]:
                t = att["text"].get("content", "")
                if t and "semantic cache" not in t: text = t
    return sql, text

def genie_query(base_url, question):
    """Fluxo completo: start-conversation -> poll -> resultado."""
    r = requests.post(f"{base_url}/api/2.0/genie/spaces/{SPACE}/start-conversation",
                      headers=H, json={"content": question}, timeout=180)
    if r.status_code == 429:
        return {"status": "429", "sql": None, "text": None}
    if r.status_code != 200:
        return {"status": f"HTTP {r.status_code}", "sql": None, "text": None}
    data = r.json()
    conv_id = data.get("conversation_id", "")
    msg_id = data.get("message_id", "")
    # App retorna tudo de uma vez
    if data.get("status") == "COMPLETED" and data.get("attachments"):
        sql, text = _extract(data)
        return {"status": "COMPLETED", "sql": sql, "text": text,
                "from_cache": conv_id.startswith("ccache_")}
    # Genie direto: precisa pollar
    for _ in range(60):
        time.sleep(2)
        r2 = requests.get(
            f"{base_url}/api/2.0/genie/spaces/{SPACE}/conversations/{conv_id}/messages/{msg_id}",
            headers=H, timeout=30)
        if r2.status_code != 200: continue
        msg = r2.json()
        if msg.get("status") == "COMPLETED":
            sql, text = _extract(msg)
            return {"status": "COMPLETED", "sql": sql, "text": text,
                    "from_cache": conv_id.startswith("ccache_")}
        if msg.get("status") in ("FAILED", "CANCELLED"):
            return {"status": msg["status"], "sql": None, "text": None}
    return {"status": "TIMEOUT", "sql": None, "text": None}

def run_parallel(label, base_url):
    """Dispara todas as queries em paralelo."""
    print(f"\n{label}")
    print("=" * 90)
    results = [None] * len(questions)
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=len(questions)) as pool:
        futs = {pool.submit(genie_query, base_url, q): i for i, q in enumerate(questions)}
        for f in as_completed(futs):
            results[futs[f]] = f.result()
    total = time.time() - t0
    for i, q in enumerate(questions):
        r = results[i]
        cache = r.get("from_cache", False)
        src = "CACHE" if cache else ("429" if r["status"]=="429" else "GENIE")
        print(f"\n[{i+1}/{len(questions)}] {src:>5s} | {r['status']}")
        print(f"  Pergunta: {q}")
        if r["sql"]:  print(f"  SQL:      {r['sql'][:100]}")
        if r["text"]: print(f"  Resposta: {r['text'][:140]}")
    print(f"\nTotal: {total:.1f}s (todas em paralelo)")
    return results, total

print("Pronto.")

## Cenario 1: Direto ao Genie (7 em paralelo)

`base_url = GENIE_HOST`. Todas disparam ao mesmo tempo. Limite: 5/min → 429.

In [ ]:
direct, d_time = run_parallel("CENARIO 1: Direto ao Genie (paralelo)", GENIE_HOST)
ok = sum(1 for r in direct if r["sql"])
blocked = sum(1 for r in direct if r["status"]=="429")
print(f"\nRESULTADO: {ok} OK, {blocked} bloqueadas (429)")

## Cenario 2a: Via App (7 em paralelo)

`base_url = APP_HOST`. **Mesmo codigo**. App gerencia rate limit + cache.

In [ ]:
app1, a1_time = run_parallel("CENARIO 2a: Via App (paralelo)", APP_HOST)
genie = sum(1 for r in app1 if not r.get("from_cache") and r["sql"])
cache = sum(1 for r in app1 if r.get("from_cache"))
print(f"\nRESULTADO: {genie} Genie, {cache} cache, 0 erros 429")

## Cenario 2b: Via App — segunda rodada (cache)

In [ ]:
app2, a2_time = run_parallel("CENARIO 2b: Via App — cache (paralelo)", APP_HOST)
print(f"\nRESULTADO: {sum(1 for r in app2 if r.get('from_cache'))}/7 cache")

## Comparacao + Cache

In [ ]:
entries = requests.get(f"{APP_HOST}/api/v1/cache", headers=H).json()
print(f"Cache: {len(entries)} entries")
for e in entries:
    print(f"  [{e['use_count']}x] {e['query_text'][:45]:45s} | {e['sql_query'][:50]}")

print(f"\n{'Query':30s} | {'Direto':>10s} | {'App 1a':>10s} | {'App 2a':>10s}")
print("-" * 70)
for i in range(len(questions)):
    d=direct[i]; a1=app1[i]; a2=app2[i]
    dc = "429" if d["status"]=="429" else ("OK" if d["sql"] else d["status"][:6])
    a1c = "CACHE" if a1.get("from_cache") else ("OK" if a1["sql"] else a1["status"][:6])
    a2c = "CACHE" if a2.get("from_cache") else a2["status"][:6]
    print(f"  {questions[i][:28]:28s} | {dc:>10s} | {a1c:>10s} | {a2c:>10s}")
print(f"  {'TEMPO TOTAL':28s} | {d_time:>9.1f}s | {a1_time:>9.1f}s | {a2_time:>9.1f}s")
print(f"\nDireto:  {sum(1 for r in direct if r['status']=='429')} queries bloqueadas (429)")
print(f"App 1a:  todas completam (cache + retry)")
print(f"App 2a:  todas do cache ({a2_time:.1f}s vs {d_time:.1f}s direto)")